# 03 — Clasificación y SHAP

Comparación Random Forest vs XGBoost y explicabilidad.


In [ ]:
import sys; sys.path.append('..')
import config, pandas as pd, numpy as np
from src import modeling
df = pd.read_csv(config.DENGUE_SEMANAL)
data = modeling.get_modeling_frame(df)
{k: v.shape for k,v in data.items() if k.startswith('X')}

## Entrenamiento y métricas en test (2024)

In [ ]:
models, info = modeling.train_models(data)
print(info)
tabla = modeling.metrics_table(models, data['X_test'], data['y_test'])
tabla

In [ ]:
mejor = modeling.elegir_mejor_modelo(tabla)
print('mejor modelo:', mejor)

## Barrido de umbral

In [ ]:
modeling.threshold_sweep(models[mejor], data['X_test'], data['y_test'])

## SHAP global (summary plot)

In [ ]:
import shap
pipe = models[mejor]
Xs = data['X_test'].sample(400, random_state=42)
Xt = modeling.transform_X(pipe, Xs)
nombres = modeling.transformed_feature_names(pipe)
expl = shap.TreeExplainer(pipe.named_steps['clf'])
sv = expl.shap_values(Xt)
sv = sv[1] if isinstance(sv, list) else (sv[:,:,1] if sv.ndim==3 else sv)
shap.summary_plot(sv, Xt, feature_names=nombres, max_display=12)

SHAP muestra asociación con la probabilidad, **no causalidad**.